# WoundScope — Colab training

這份 notebook 只負責 orchestration；training、evaluation、calibration 與 ONNX 邏輯都來自 `src/woundscope`。預設 quick mode 目標為常見 T4 16 GB。所有 run artifacts 直接寫入 Google Drive，每個 epoch 都能 resume。

> 輸出僅為研究用 wound segmentation，不是疾病診斷、嚴重度或治療建議。FUSeg 原圖、labels、manifest 與 gallery 不得公開重傳。

In [ ]:
#@title 1. Locked run settings
RUN_MODE = "quick"  #@param ["quick", "full"]
FULL_STAGE = "comparison"  #@param ["comparison", "final"]
CROSS_SPLIT_POLICY = "error"  #@param ["error", "exclude_train"]
PROJECT_GIT_URL = ""  # optional, leave blank before GitHub publication
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/WoundScope"
DRIVE_PROJECT_ZIP = "/content/drive/MyDrive/WoundScope_colab_source.zip"
DRIVE_ARTIFACT_DIR = "/content/drive/MyDrive/WoundScopeArtifacts"
SELECTED_LOSS_UNET = "bce_dice"  # set only after full comparison
SELECTED_LOSS_SEGFORMER = "bce_dice"  # set only after full comparison
assert RUN_MODE in {"quick", "full"}
assert CROSS_SPLIT_POLICY in {"error", "exclude_train"}

In [ ]:
#@title 2. Mount Drive and load project
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, shutil, subprocess
project_dir = Path('/content/WoundScope')
if PROJECT_GIT_URL:
    if not project_dir.exists():
        subprocess.run(['git', 'clone', '--depth', '1', PROJECT_GIT_URL, str(project_dir)], check=True)
else:
    source_dir = Path(DRIVE_PROJECT_DIR)
    source_zip = Path(DRIVE_PROJECT_ZIP)
    if project_dir.exists():
        shutil.rmtree(project_dir)
    if (source_dir / 'pyproject.toml').is_file():
        shutil.copytree(source_dir, project_dir, ignore=shutil.ignore_patterns('.venv', 'data', 'artifacts', '.git'))
    elif source_zip.is_file():
        shutil.unpack_archive(source_zip, project_dir)
    else:
        raise FileNotFoundError(f'請上傳 source folder 到 {source_dir}，或 ZIP 到 {source_zip}')
os.chdir(project_dir)
os.environ['WOUNDSCOPE_DATA_DIR'] = '/content/woundscope_data'
os.environ['WOUNDSCOPE_ARTIFACT_DIR'] = DRIVE_ARTIFACT_DIR
Path(DRIVE_ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)
print('Project:', project_dir)
print('Persistent artifacts:', DRIVE_ARTIFACT_DIR)

In [ ]:
#@title 3. Install and verify GPU
%pip install -q -e ".[train,export,app]"
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('請在 Runtime > Change runtime type 選 GPU 後重新執行。')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
#@title 4. Download pinned official FUSeg and inspect integrity
import json, subprocess, sys
subprocess.run([sys.executable, 'scripts/download_data.py', '--allow-cross-split-exact'], check=True)
summary_path = Path(os.environ['WOUNDSCOPE_DATA_DIR']) / 'manifests/data_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('Official counts:', summary['counts'])
print('Internal counts:', summary['internal_counts'])
print('Exact cross-split findings:', len(summary['exact_cross_split']))
for finding in summary['exact_cross_split']:
    print(finding['samples'])
if CROSS_SPLIT_POLICY == 'error':
    raise RuntimeError('安全停止：請先審閱上方 findings。若同意排除 train copies，把 CROSS_SPLIT_POLICY 改成 exclude_train，再 Run all。')

## Experiment protocol

- Quick：兩模型 × 兩 losses，各 2 epochs、固定 subset，只是 internal-dev smoke。
- Full comparison：seed 42 跑兩模型 × 兩 losses。
- Full final：先依 comparison 選好每架構 loss，再跑 seeds 42/43/44。不可用 official validation 選 loss。

In [ ]:
#@title 5. Build experiment matrix
model_configs = {
    'unet_efficientnet_b0': 'configs/models/unet_efficientnet_b0.yaml',
    'segformer_b0': 'configs/models/segformer_b0.yaml',
}
if RUN_MODE == 'quick':
    experiments = [(name, path, loss, 42) for name, path in model_configs.items() for loss in ('bce_dice', 'focal_tversky')]
elif FULL_STAGE == 'comparison':
    experiments = [(name, path, loss, 42) for name, path in model_configs.items() for loss in ('bce_dice', 'focal_tversky')]
else:
    selected = {'unet_efficientnet_b0': SELECTED_LOSS_UNET, 'segformer_b0': SELECTED_LOSS_SEGFORMER}
    experiments = [(name, path, selected[name], seed) for name, path in model_configs.items() for seed in (42, 43, 44)]
print(*experiments, sep='\n')

In [ ]:
#@title 6. Train/resume; every epoch persists to Drive
mode_config = f'configs/modes/{RUN_MODE}.yaml'
run_dirs = []
for model_name, model_config, loss, seed in experiments:
    run_dir = Path(DRIVE_ARTIFACT_DIR) / 'runs' / f'{RUN_MODE}_{FULL_STAGE}_{model_name}_{loss}_seed{seed}'
    run_dirs.append((run_dir, model_config, loss, seed))
    command = [
        sys.executable, 'scripts/train.py', '--model-config', model_config,
        '--mode-config', mode_config, '--device', 'cuda', '--run-dir', str(run_dir),
        '--cross-split-policy', CROSS_SPLIT_POLICY, '--resume',
        '--set', f'training.loss={loss}', '--set', f'project.seed={seed}',
    ]
    print('Running:', model_name, loss, seed)
    subprocess.run(command, check=True)

In [ ]:
#@title 7. Dev calibration, evaluation, ONNX and sample prediction
import csv
manifest_path = Path(os.environ['WOUNDSCOPE_DATA_DIR']) / 'manifests/data_manifest.csv'
with manifest_path.open(encoding='utf-8', newline='') as handle:
    dev_row = next(row for row in csv.DictReader(handle) if row['internal_split'] == 'dev')
challenge_dir = Path(os.environ['WOUNDSCOPE_DATA_DIR']) / 'raw/fuseg/data/Foot Ulcer Segmentation Challenge'
for run_dir, model_config, loss, seed in run_dirs:
    checkpoint = run_dir / 'best_model.safetensors'
    calibration = run_dir / 'calibration.json'
    common = ['--model-config', model_config, '--mode-config', mode_config, '--checkpoint', str(checkpoint), '--calibration', str(calibration), '--device', 'cuda', '--set', f'training.loss={loss}', '--set', f'project.seed={seed}']
    subprocess.run([sys.executable, 'scripts/evaluate.py', *common, '--selector', 'dev', '--fit-calibration', '--output', str(run_dir / 'dev_evaluation')], check=True)
    if RUN_MODE == 'full' and FULL_STAGE == 'final':
        subprocess.run([sys.executable, 'scripts/evaluate.py', *common, '--selector', 'official_validation', '--output', str(run_dir / 'official_validation')], check=True)
    onnx_path = run_dir / 'model.onnx'
    subprocess.run([sys.executable, 'scripts/export_onnx.py', '--model-config', model_config, '--checkpoint', str(checkpoint), '--output', str(onnx_path)], check=True)
    sample_dir = run_dir / 'sample_predictions'
    subprocess.run([sys.executable, 'scripts/predict.py', '--model', str(onnx_path), '--calibration', str(calibration), '--input', str(challenge_dir / dev_row['image_relpath']), '--output', str(sample_dir), '--device', 'cpu'], check=True)
print('All artifacts persisted under', DRIVE_ARTIFACT_DIR)

In [ ]:
#@title 8. Inspect logs (optional)
%load_ext tensorboard
%tensorboard --logdir $DRIVE_ARTIFACT_DIR/runs

## Handoff

確認每個 run 至少含 `best_model.safetensors`、`last_model.safetensors`、`trainer_state.pt`、`history.csv`、`results.json`、`provenance.json`、`calibration.json`、`model.onnx`、evaluation 與 sample predictions。下載到 WSL 的步驟見 `scripts/download_artifacts.md`。不要把 FUSeg data、manifest 或 gallery 上傳到公開 repository。